# 02 — Greeks: analytic vs bump-and-reprice

Two ways to get sensitivities:

1. **Analytic** (`bs_greeks`) — exact, fast, but only exists for the simplest models.
2. **Finite differences** (`finite_difference_greeks`) — works for *any* pricer
   that takes `S, K, T, r, sigma` as keywords. Less accurate, universally applicable.

Here we check that the second reproduces the first, then look at what the Greeks
look like across spot, and finally see the trap: finite-differencing a **tree**
pricer amplifies its discretisation noise, especially for gamma.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optlab.pricing.black_scholes import bs_price, bs_greeks
from optlab.pricing.binomial import binomial_price
from optlab.pricing.monte_carlo import mc_price

from optlab.greeks.finite_diff import finite_difference_greeks, greeks_error_table

## 1. Agreement on a grid of contracts

Central differences with a relative bump of `1e-4` reproduce the analytic values
to well under 1e-4 absolute. Gamma is the worst of the set — it is a *second*
difference, so the bump appears squared in the denominator and the truncation
error is larger.

In [ ]:
rows = []
for S0, K0, T0, r0, sig0, kind in [
    (100, 100, 1.00, 0.05, 0.20, "call"),
    (100,  80, 0.50, 0.03, 0.25, "call"),
    (100, 120, 0.25, 0.02, 0.30, "put"),
    (50,   50, 2.00, 0.04, 0.15, "put"),
]:
    analytic = bs_greeks(S0, K0, T0, r0, sig0, 0.0, kind)
    numeric = finite_difference_greeks(bs_price, S=S0, K=K0, T=T0, r=r0, sigma=sig0, q=0.0, kind=kind)
    errs = greeks_error_table(analytic, numeric)
    rows.append({"S": S0, "K": K0, "T": T0, "sigma": sig0, "kind": kind,
                 **{f"FD {k}": getattr(numeric, k) for k in errs},
                 **{f"err {k}": e for k, e in errs.items()}})
pd.DataFrame(rows).round(6).T

## 2. Delta, gamma, vega and theta across spot

The shape is the whole story of an option book:

* **delta** saturates at 1 (deep ITM) and 0 (deep OTM), with maximum slope at the money;
* **gamma** peaks at the money — that is why at-the-money options are the expensive ones to hedge;
* **vega** peaks at the money too, so a strike away from spot has very little vol risk;
* **theta** is most negative at the money, where time value is largest.

In [ ]:
spots = np.linspace(60, 140, 161)
greeks = bs_greeks(spots, 100.0, 0.5, 0.03, 0.2)

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
for ax, name, label in zip(
    axes.ravel(),
    ["delta", "gamma", "vega", "theta"],
    ["dV/dS", "d2V/dS2", "dV/dsigma (per 1.00 vol)", "dV/dt (per year)"],
):
    ax.plot(spots, getattr(greeks, name))
    ax.axvline(100.0, color="grey", linestyle=":", linewidth=1)
    ax.set_title(f"{name} — {label}")
    ax.set_xlabel("spot")
fig.suptitle("Long at-the-money call, T = 0.5y, sigma = 20%")
fig.tight_layout()
plt.show()

## 3. The trap: finite-differencing a lattice

Gamma from the tree is a second difference of a price that is only accurate to
~`1/steps`. The error gets divided by `h^2`, so it *grows* as the bump shrinks.
This is the classic bump-size trade-off: too small and noise dominates, too large
and truncation error dominates.

Try changing `rel_bump` and watch gamma fall apart.

In [ ]:
analytic_gamma = bs_greeks(100, 100, 1.0, 0.05, 0.2).gamma
rows = []
for steps in [200, 600, 2000]:
    for bump in [1e-3, 1e-2, 5e-2]:
        g = finite_difference_greeks(
            binomial_price, S=100, K=100, T=1.0, r=0.05, sigma=0.2,
            rel_bump=bump, pricer_kwargs={"steps": steps},
        )
        rows.append({"steps": steps, "rel_bump": bump, "FD gamma": g.gamma,
                     "analytic": analytic_gamma, "error": abs(g.gamma - analytic_gamma)})
pd.DataFrame(rows).round(5)

## Takeaway

Use analytic Greeks when you have them and finite differences when you do not —
but treat lattice and Monte Carlo Greeks as *estimates with their own error*, and
bump large enough to dominate the pricer's discretisation noise. Hedging with a
gamma estimate that is off by 10% is worse than not hedging at all.